# Produttivita italiana: PIL, occupati, ore e settori ISTAT

Questo notebook usa i file ISTAT locali presenti nel repository.  
Serve per controllare due fenomeni distinti: il PIL che cresce meno delle ore lavorate e la produttivita implicita per settore.

In [ ]:
from pathlib import Path
import sys

# Il notebook puo essere eseguito dalla root del repo oppure dalla cartella notebooks.
ROOT = Path.cwd()
if not (ROOT / "analisi").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from analisi.pil.pil_lavoro import build_panel, plot_and_save, OUTDIR as PIL_OUT

# Costruisce indici trimestrali base 2015=100: PIL, occupati, ore e produttivita implicita.
pil_panel = build_panel()
plot_and_save(pil_panel)

pil_panel.tail(12)

## Lettura rapida

Se `prod_per_hour_idx` scende mentre PIL e ore salgono, significa che il lavoro aggiuntivo non si sta traducendo in valore aggiunto proporzionale.

In [ ]:
# Ultimo dato disponibile con produttivita per ora non nulla.
pil_panel.dropna(subset=["prod_per_hour_idx"]).tail(1)

In [ ]:
for img in [
    PIL_OUT / "pil_vs_occupati_2015_100.png",
    PIL_OUT / "pil_vs_ore_lavorate_2015_100.png",
    PIL_OUT / "produttivita_implicita_2015_100.png",
]:
    display(Image(filename=str(img)))

## Settori: valore aggiunto vs ore lavorate

Lo script `valore_aggiunto_pil.py` allinea valore aggiunto annuale e ore lavorate per branca ATECO.  
Qui leggiamo gli output gia generati e guardiamo i settori con produttivita implicita piu debole nell'ultimo anno.

In [ ]:
VA_OUT = ROOT / "analisi" / "valore_aggiunto" / "output"
summary_sectors = pd.read_csv(VA_OUT / "summary_sectors_va_gt_hours_base2021.csv")

cols = ["sector_va", "last_year_check", "va_idx_last", "hours_idx_last", "prod_idx_last", "gap_va_minus_hours_last"]
summary_sectors.sort_values("prod_idx_last")[cols].head(12)

In [ ]:
# Mostra alcuni grafici settoriali gia prodotti dalla pipeline.
plots_dir = VA_OUT / "plots_all_sectors"
selected = [
    "prod_idx_istruzione.png",
    "prod_idx_servizi_di_alloggio_e_di_ristorazione.png",
    "prod_idx_commercio_allingrosso_e_al_dettaglio_riparazione_di_autoveicoli_e_motocicli.png",
    "prod_idx_industria_manifatturiera.png",
]
for name in selected:
    path = plots_dir / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Manca:", path.name)